# Phi3V-Med baseline — 체크포인트에서 이어서 학습

`Phi3V_baseline_colab.ipynb`로 학습하다 세션이 끊겼을 때 쓰는 노트북.
Drive의 `phi3_baseline/work/lora_{ds}/checkpoint-*` 중 **마지막 정상 체크포인트**에서 학습을 재개하고, 끝나면 추론·평가까지 진행한다.

- 재개 시 precision / crops / epochs / lr / batch / seed는 **원래 실행의 `run_config.json` 값을 그대로** 사용한다 (값이 바뀌면 스케줄러·스텝 수가 어긋나므로).
- 저장 도중 끊겨 깨진 체크포인트는 `broken-checkpoint-*`로 이름을 바꿔 건너뛴다.
- 학습이 이미 끝난 데이터셋(`adapter_model.safetensors` 존재)은 건너뛴다.

**사용법**: 런타임 → 런타임 유형 변경 → 원래 학습과 같은 종류의 GPU 선택 후 위에서부터 실행.

## 1. 설정

In [1]:
DATASETS = ['vqa_rad', 'slake']
LIMIT    = None   # 스모크 테스트용 test 질문 수. 전체 실행은 None

# 체크포인트가 없는 데이터셋을 새로 학습할 때만 쓰는 값 (재개 시에는 run_config.json 값 사용)
NEW_PREC        = None   # None이면 GPU에 맞게 자동 (A100/L4: bf16, T4: fp16)
NEW_TRAIN_CROPS = 16
NEW_EPOCHS      = 10

BASE = '/content/drive/MyDrive/phi3_baseline'
DATA = f'{BASE}/data'
WORK = f'{BASE}/work'
CODE = '/content/phi3_baseline'

## 2. Drive 연결, GPU 확인, 패키지 설치
무거운 작업은 모두 `!python -m ...` 하위 프로세스로 실행하므로 설치 후 런타임 재시작이 필요 없다.

In [2]:
import os, shutil, sys
from google.colab import drive
drive.mount('/content/drive')
os.makedirs(f'{CODE}/phi3cfg', exist_ok=True)
os.makedirs(WORK, exist_ok=True)
!nvidia-smi --query-gpu=name,memory.total --format=csv
print(sys.version)
# Colab Python 3.13: tokenizers 0.19.x (transformers 4.44.2 의존)는 cp313 wheel이 없어 소스 빌드됨 -> Rust 필요
CARGO_BIN = os.path.expanduser('~/.cargo/bin')
if not shutil.which('cargo') and not os.path.exists(f'{CARGO_BIN}/cargo'):
    !curl -sSf https://sh.rustup.rs | sh -s -- -y --profile minimal -q
os.environ['PATH'] = CARGO_BIN + ':' + os.environ['PATH']
!pip install transformers==4.44.2 peft==0.12.0 accelerate==0.34.2 "bitsandbytes>=0.44.1" \
    huggingface-hub==0.25.2 datasets==2.21.0 scikit-learn

Mounted at /content/drive
name, memory.total [MiB]
NVIDIA A100-SXM4-80GB, 81920 MiB
3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]

  stable-x86_64-unknown-linux-gnu installed - rustc 1.100.0 (0ef1bc386 2026-11-09)


Rust is installed now. Great!

To get started you may need to restart your current shell.
This would reload your PATH environment variable to include
Cargo's bin directory ($HOME/.cargo/bin).

To configure your current shell, you need to source the
corresponding env file under $HOME/.cargo.

Consider running the right command for your shell (note the leading DOT):
. "$HOME/.cargo/env" # For sh/ash/dash/pdksh/bash
cargo:rerun-if-env-changed=CC_x86_64-unknown-linux-gnu
CC_x86_64-unknown-linux-gnu = None
cargo:rerun-if-env-changed=CC_x86_64_unknown_linux_gnu
CC_x86_64_unknown_linux_gnu = None
cargo:rerun-if-env-changed=HOST_CC
HOST_CC = None
cargo:rerun-if-env-changed=CC
CC = None
cargo:rerun-if-env-changed=CC_ENABLE_DEBUG_OUTPUT
cargo:rerun-if-env-changed=CPATH
cargo:rer

## 3. 코드 파일 생성
셀을 펼치면 코드가 보인다.

In [ ]:
%%writefile /content/phi3_baseline/phi3cfg/__init__.py
#@title phi3cfg/__init__.py
"""Phi-3-Vision (Phi3V-Med, 4.2B) reproduction of Expert-CFG (arXiv:2507.09209)."""


In [ ]:
%%writefile /content/phi3_baseline/phi3cfg/data.py
#@title phi3cfg/data.py
import hashlib
import json
from pathlib import Path

DATASETS = ('vqa_rad', 'slake', 'pathvqa')


def read_jsonl(path):
    with open(path, encoding='utf-8') as f:
        return [json.loads(line) for line in f if line.strip()]


def digest(path):
    return hashlib.sha256(Path(path).read_bytes()).hexdigest()


def load_data(path, split=None):
    rows = read_jsonl(path)
    if not rows:
        raise ValueError('Empty dataset')
    seen = set()
    for r in rows:
        for key in ('id', 'dataset', 'split', 'image', 'question', 'answer', 'answer_type'):
            if key not in r:
                raise ValueError(f'Missing {key}')
        key = (r['dataset'], str(r['id']))
        if key in seen:
            raise ValueError(f'Duplicate id: {key}')
        seen.add(key)
        if r['dataset'] not in DATASETS or r['answer_type'] not in ('open', 'closed'):
            raise ValueError('Invalid dataset or answer_type')
        if split and r['split'] != split:
            raise ValueError(f'Expected {split}, got {r["split"]}')
        if not str(r['answer']).strip():
            raise ValueError('Empty answer')
    return rows


def audit_splits(train, test):
    if {(r['dataset'], str(r['id'])) for r in train} & {(r['dataset'], str(r['id'])) for r in test}:
        raise ValueError('Train/test question ID overlap')
    # Image overlap is reported, not rejected: some MedVQA splits share images.
    images = {(r['dataset'], r['image']) for r in train} & {(r['dataset'], r['image']) for r in test}
    return {'shared_images': len(images), 'train': len(train), 'test': len(test)}


In [ ]:
%%writefile /content/phi3_baseline/phi3cfg/download_data.py
#@title phi3cfg/download_data.py
"""Download HF MedVQA mirrors and export the existing baseline JSONL/image interface."""
import argparse
import hashlib
import io
import json
import os
from pathlib import Path, PurePosixPath
import shutil
import tarfile
import tempfile
import zipfile

from PIL import Image
from .data import digest, load_data

SOURCES = {
    'vqa_rad': 'flaviagiammarino/vqa-rad',
    'slake': 'BoKelvin/SLAKE',
    'pathvqa': 'flaviagiammarino/path-vqa',
}
IMAGE_EXT = {'.png', '.jpg', '.jpeg', '.bmp', '.webp', '.tif', '.tiff'}
FORMAT_VERSION = 1


def get_field(row, keys, default=None):
    for key in keys:
        if key in row and row[key] is not None:
            return row[key]
    return default


class InvalidAnnotation(ValueError):
    """A source row with unusable content, not a missing/unsupported schema."""


def normalize(row, dataset, split, index):
    language = get_field(row, ('q_lang', 'language', 'lang'))
    if dataset == 'slake' and language is not None:
        if str(language).strip().lower() not in ('en', 'eng', 'english'):
            return None
    question = get_field(row, ('question', 'query'))
    answer = get_field(row, ('answer', 'label'))
    if not any(k in row for k in ('question', 'query')) or not any(k in row for k in ('answer', 'label')):
        raise ValueError(f'{dataset}/{split} row {index}: question/answer columns missing; columns={list(row)}')
    if not isinstance(question, str) or not question.strip():
        raise InvalidAnnotation('empty_or_nontext_question')
    if answer is None or not str(answer).strip():
        raise InvalidAnnotation('empty_answer')
    answer = str(answer).strip()
    if answer.lower() in ('yes', 'no'):
        answer = answer.lower()
    kind = get_field(row, ('answer_type', 'question_answer_type'))
    if kind is None:
        kind = 'closed' if answer.lower() in ('yes', 'no') else 'open'
    else:
        kind = str(kind).strip().lower()
        kind = {'close': 'closed', 'closed-ended': 'closed', 'open-ended': 'open'}.get(kind, kind)
        if kind not in ('closed', 'open'):
            raise ValueError(f'Unsupported answer_type={kind!r}; do not guess numeric category encodings')
    source_id = str(get_field(row, ('qid', 'question_id', 'q_id', 'id'), index))
    # Mirror IDs can be local to a split; preserve the original separately.
    return {'id': f'{split}:{source_id}', 'source_id': source_id, 'source_row': index,
            'dataset': dataset, 'split': split, 'question': question,
            'answer': answer, 'answer_type': kind}


def open_image(value, resolver=None):
    if isinstance(value, Image.Image):
        return value.convert('RGB')
    if isinstance(value, dict):
        if value.get('bytes') is not None:
            with Image.open(io.BytesIO(value['bytes'])) as im:
                return im.convert('RGB')
        value = value.get('path')
    if isinstance(value, (str, Path)):
        path = Path(value)
        if not path.is_file() and resolver:
            path = Path(resolver(str(value)))
        if path.is_file():
            with Image.open(path) as im:
                return im.convert('RGB')
    raise ValueError(f'Image not embedded or not found: {str(value)[:160]}')


def cached_paths(target, source, revision=None, config=None):
    path = target/'provenance.json'
    if not path.exists():
        return None
    manifest = json.loads(path.read_text())
    expected = (source, config, FORMAT_VERSION)
    actual = (manifest['source'], manifest.get('config'), manifest.get('format_version'))
    if actual != expected or (revision and revision not in (manifest['revision'], manifest.get('requested_revision'))):
        raise ValueError(f'{target}: source/config/revision changed; use a different --output directory')
    for file, checksum in manifest['files'].items():
        p = target/file
        if not p.is_file() or digest(p) != checksum:
            raise ValueError(f'Prepared data missing/modified: {p}; use a new output directory to rebuild')
    for filename in manifest['images']:
        image = target/'images'/filename
        if not image.is_file() or digest(image) != manifest['images'][filename]:
            raise ValueError(f'Prepared image missing/modified: {filename}; use a new output directory to rebuild')
    return {'images': str(target/'images'), **{s: str(target/f'{s}.jsonl') for s in manifest['splits']}}


def export(dataset, splits, target, source, revision, config=None,
           requested_revision=None, resolver=None):
    """Publish only a fully exported directory; retain HF's train/test/validation splits."""
    target = Path(target).resolve()
    if 'train' not in splits or 'test' not in splits:
        raise ValueError(f'{source} must supply train AND test; available splits={list(splits)}. No random split is created.')
    if target.exists():
        raise ValueError(f'Refusing to overwrite existing directory: {target}')
    target.parent.mkdir(parents=True, exist_ok=True)
    staging = Path(tempfile.mkdtemp(prefix=f'.{dataset}-', dir=target.parent))
    try:
        (staging/'images').mkdir()
        image_hashes = {}
        manifest = {'format_version': FORMAT_VERSION, 'dataset': dataset,
                    'source': source, 'revision': revision, 'requested_revision': requested_revision,
                    'config': config, 'english_only': dataset == 'slake',
                    'split_policy': 'Preserve mirror splits; no random resplit; not verified identical to paper',
                    'annotation_policy': 'Quarantine empty/null answers and empty/nontext questions; never impute ground truth',
                    'splits': {}, 'files': {}, 'images': image_hashes}
        for source_split, records in splits.items():
            split = 'validation' if source_split in ('val', 'valid', 'dev') else source_split
            if split not in ('train', 'validation', 'test'):
                continue
            if split in manifest['splits']:
                raise ValueError(f'Multiple source splits map to {split}')
            raw_count = count = invalid_count = language_count = 0
            closed = 0
            columns = None
            rejected_path = staging/f'{split}.rejected.jsonl'
            with (staging/f'{split}.jsonl').open('w', encoding='utf-8') as f, rejected_path.open('w', encoding='utf-8') as rejected:
                for index, row in enumerate(records):
                    raw_count += 1
                    columns = columns or sorted(row)
                    try:
                        out = normalize(row, dataset, split, index)
                    except InvalidAnnotation as error:
                        invalid_count += 1
                        audit = {'dataset': dataset, 'source_split': source_split, 'source_row': index,
                                 'source_id': str(get_field(row, ('qid','question_id','q_id','id'), index)),
                                 'reason': str(error), 'question': get_field(row, ('question','query')),
                                 'answer': get_field(row, ('answer','label')),
                                 'image_reference': str(get_field(row, ('img_name','image_name','image_path','img_id'), 'embedded')),
                                 'language': get_field(row, ('q_lang','language','lang'))}
                        rejected.write(json.dumps(audit, ensure_ascii=False, default=str)+'\n')
                        continue
                    if out is None:
                        language_count += 1
                        continue
                    value = get_field(row, ('image', 'img', 'img_name', 'image_name', 'image_path', 'img_id'))
                    image = open_image(value, resolver)
                    # Same pixels share a lossless image, including across train/test.
                    identity = hashlib.sha256(str(image.size).encode()+image.tobytes()).hexdigest()
                    filename = identity+'.png'
                    if filename not in image_hashes:
                        image.save(staging/'images'/filename, format='PNG')
                        image_hashes[filename] = digest(staging/'images'/filename)
                    out['image'] = filename
                    out['source_split'] = source_split
                    f.write(json.dumps(out, ensure_ascii=False)+'\n')
                    count += 1
                    closed += out['answer_type'] == 'closed'
            if count == 0:
                raise ValueError(f'{source}/{source_split}: no usable questions after filtering')
            load_data(staging/f'{split}.jsonl', split)
            manifest['splits'][split] = {'source_split': source_split, 'raw': raw_count,
                                        'kept': count, 'open': count-closed, 'closed': closed,
                                        'columns': columns, 'filtered_language': language_count,
                                        'invalid_annotations': invalid_count,
                                        'rejected_file': rejected_path.name}
            manifest['files'][f'{split}.jsonl'] = digest(staging/f'{split}.jsonl')
            manifest['files'][rejected_path.name] = digest(rejected_path)
            print(f'{dataset}/{split}: kept {count}/{raw_count}; language excluded={language_count}; '
                  f'invalid annotations={invalid_count} (audit: {rejected_path.name})', flush=True)
        # Sharing images is allowed; record exact duplicate questions without changing the source split.
        train = load_data(staging/'train.jsonl', 'train')
        test = load_data(staging/'test.jsonl', 'test')
        keys = lambda rows: {(r['image'], r['question'].strip().lower()) for r in rows}
        manifest['train_test_duplicate_questions'] = len(keys(train) & keys(test))
        manifest['train_test_shared_images'] = len({r['image'] for r in train} & {r['image'] for r in test})
        (staging/'provenance.json').write_text(json.dumps(manifest, indent=2, ensure_ascii=False)+'\n')
        staging.rename(target)
    finally:
        if staging.exists():
            shutil.rmtree(staging)
    return cached_paths(target, source, revision, config)


def unpack_assets(archives, destination):
    """Only regular image/JSON assets; never extract links or archive traversal paths."""
    destination = Path(destination)
    destination.mkdir(parents=True, exist_ok=True)
    def copy_member(name, stream):
        part = PurePosixPath(name.replace('\\', '/'))
        if part.is_absolute() or '..' in part.parts:
            raise ValueError(f'Unsafe archive member: {name}')
        if part.suffix.lower() not in IMAGE_EXT | {'.json'}:
            return
        out = destination.joinpath(*part.parts)
        out.parent.mkdir(parents=True, exist_ok=True)
        with out.open('wb') as f:
            shutil.copyfileobj(stream, f)
    for archive in archives:
        if zipfile.is_zipfile(archive):
            with zipfile.ZipFile(archive) as z:
                for member in z.infolist():
                    if member.is_dir():
                        continue
                    if (member.external_attr >> 16) & 0o170000 == 0o120000:
                        raise ValueError('Archive symlinks are unsupported')
                    with z.open(member) as stream:
                        copy_member(member.filename, stream)
        elif tarfile.is_tarfile(archive):
            with tarfile.open(archive) as t:
                for member in t:
                    if member.isfile():
                        with t.extractfile(member) as stream:
                            copy_member(member.name, stream)
        else:
            raise ValueError(f'Unsupported image archive: {archive}')


def asset_resolver(repo, revision, cache_dir):
    """Lazy fallback for mirrors with img_name instead of embedded HF Image columns."""
    roots = []
    by_basename = None
    def resolve(name):
        nonlocal by_basename
        if not roots:
            from huggingface_hub import snapshot_download
            root = Path(snapshot_download(repo, repo_type='dataset', revision=revision, cache_dir=cache_dir,
                         allow_patterns=['*.png','*.jpg','*.jpeg','*.PNG','*.JPG','*.json','*.zip','*.tar','*.tar.gz','*.tgz']))
            roots.append(root)
            archives = [p for p in root.rglob('*') if p.is_file() and
                        (p.name.lower().endswith(('.zip','.tar','.tar.gz','.tgz')))]
            if archives:
                extracted = Path(cache_dir or tempfile.gettempdir())/'medvqa_assets'/revision
                unpack_assets(archives, extracted)
                roots.append(extracted)
        relative = PurePosixPath(name.replace('\\','/'))
        if relative.is_absolute() or '..' in relative.parts or not relative.parts:
            raise ValueError(f'{repo}: invalid relative image path: {name!r}')
        for root in roots:
            direct = root.joinpath(*relative.parts)
            if direct.is_file():
                return str(direct)
        if by_basename is None:
            # Index once, rather than traversing all assets for every question.
            by_basename = {}
            for root in roots:
                for path in root.rglob('*'):
                    if path.is_file() and path.suffix.lower() in IMAGE_EXT:
                        by_basename.setdefault(path.name, []).append(path)
        candidates = by_basename.get(relative.name, [])
        # SLAKE uses xmlab1/source.jpg, xmlab2/source.jpg, ... . Archive
        # prefixes may vary, but the complete annotated relative path must match.
        # Compare before resolve(): HF cache symlinks resolve to extensionless blobs.
        matches = [p for p in candidates if p.parts[-len(relative.parts):] == relative.parts]
        unique = sorted({p.resolve() for p in matches})
        if len(unique) != 1:
            raise ValueError(f'{repo}: image {name!r} has {len(unique)} full-path matches '
                             f'({len(candidates)} basename candidates). '
                             'Source may not include the required images or has ambiguous archive paths.')
        return str(unique[0])
    return resolve


def load_raw_splits(repo, revision, cache_dir):
    """Fallback for JSON/image archives without a load_dataset-compatible layout."""
    from huggingface_hub import snapshot_download
    root = Path(snapshot_download(repo, repo_type='dataset', revision=revision, cache_dir=cache_dir,
                                 allow_patterns=['*.json','*.zip','*.tar','*.tar.gz','*.tgz']))
    roots = [root]
    archives = [p for p in root.rglob('*') if p.is_file() and
                p.name.lower().endswith(('.zip','.tar','.tar.gz','.tgz'))]
    if archives:
        extracted = Path(cache_dir or tempfile.gettempdir())/'medvqa_raw'/revision
        unpack_assets(archives, extracted)
        roots.append(extracted)
    splits = {}
    for split, filenames in {'train':('train.json',), 'test':('test.json',),
                             'validation':('validation.json','val.json','valid.json','dev.json')}.items():
        candidates = sorted({p.resolve() for root in roots for name in filenames
                             for p in root.rglob(name) if p.is_file()})
        if len(candidates) > 1:
            raise ValueError(f'{repo}: ambiguous {split} JSON annotations: {len(candidates)} files')
        if candidates:
            records = json.loads(candidates[0].read_text(encoding='utf-8'))
            if isinstance(records, dict):
                records = records.get('data', records.get('annotations'))
            if not isinstance(records, list):
                raise ValueError(f'{repo}: {split} annotation must be a JSON array')
            splits[split] = records
    if not {'train','test'} <= splits.keys():
        raise ValueError(f'{repo}: no unambiguous train.json/test.json fallback found')
    return splits


def download_one(dataset, output, source=None, revision=None, config=None, cache_dir=None):
    source = source or SOURCES[dataset]
    target = Path(output).resolve()/dataset
    existing = cached_paths(target, source, revision, config)
    if existing:
        print(f'{dataset}: reusing prepared Drive data ({target})', flush=True)
        return existing
    if target.exists():
        raise ValueError(f'{target} exists without a complete provenance manifest; use a different output root')
    try:
        from datasets import load_dataset, Image as HFImage
        from huggingface_hub import HfApi
    except ImportError as error:
        raise RuntimeError('Install requirements-colab.txt first (datasets==2.21.0 required)') from error
    try:
        sha = HfApi().dataset_info(source, revision=revision or 'main').sha
        print(f'Downloading {dataset}: {source}@{sha}', flush=True)
        try:
            splits = load_dataset(source, name=config, revision=sha, cache_dir=cache_dir, trust_remote_code=False)
        except (ValueError, RuntimeError, FileNotFoundError) as load_error:
            if config:
                raise
            print(f'Parquet/standard loader unavailable: {load_error}. Trying raw train/test JSON.', flush=True)
            splits = load_raw_splits(source, sha, cache_dir)
        print(f'{source}: splits={list(splits)}', flush=True)
        # Decode with Pillow ourselves; support path and byte payloads consistently.
        for key in list(splits):
            for column, feature in getattr(splits[key], 'features', {}).items():
                if isinstance(feature, HFImage):
                    splits[key] = splits[key].cast_column(column, HFImage(decode=False))
        return export(dataset, splits, target, source, sha, config, revision,
                      resolver=asset_resolver(source, sha, cache_dir))
    except Exception as error:
        raise RuntimeError(f'{dataset}: HF source {source} could not be prepared: {error}. '
                           'No alternative split or dataset was silently substituted. '
                           'Check the dataset card/access or use --source with a compatible mirror.') from error


def main():
    p = argparse.ArgumentParser(description=__doc__)
    p.add_argument('--output', required=True, help='Persistent prepared dataset root (e.g. Drive/urp_data/hf)')
    p.add_argument('--datasets', nargs='+', choices=tuple(SOURCES), default=list(SOURCES))
    p.add_argument('--cache-dir', help='Optional HF download/cache directory')
    p.add_argument('--source', help='Override HF repo ID; use with exactly one dataset')
    p.add_argument('--revision', help='Pin a HF source commit/ref; exactly one dataset')
    p.add_argument('--config', help='HF dataset config; exactly one dataset')
    args = p.parse_args()
    if len(args.datasets) != 1 and any((args.source, args.revision, args.config)):
        p.error('--source/--revision/--config require exactly one --datasets entry')
    paths = {}
    for dataset in args.datasets:
        paths[dataset] = download_one(dataset, args.output, args.source, args.revision, args.config, args.cache_dir)
    path = Path(args.output)/'data_paths.json'
    temp = path.with_suffix('.tmp')
    temp.write_text(json.dumps(paths, indent=2)+'\n')
    os.replace(temp, path)
    print(f'Ready: {path}', flush=True)


if __name__ == '__main__':
    main()


In [ ]:
%%writefile /content/phi3_baseline/phi3cfg/model.py
#@title phi3cfg/model.py
"""Phi-3-Vision (4.2B) + Phi-3V-Med LoRA loader. Phi-3.5 is intentionally not supported."""
import importlib.metadata

import torch

BASE_MODEL = 'microsoft/Phi-3-vision-128k-instruct'
MED_ADAPTER = 'ecoxial2007/Phi-3V-Med'


def patch_clip_for_lora(model):
    """Author's patch (CheX-Phi35V): drop unused CLIP layers. Output is unchanged."""
    embedder = next(m for n, m in model.named_modules() if n.endswith('vision_embed_tokens'))

    def get_img_features(self, img_embeds):
        vm = self.img_processor.vision_model
        h = vm.pre_layrnorm(vm.embeddings(img_embeds))
        return vm.encoder(inputs_embeds=h, output_hidden_states=True).hidden_states[-1][:, 1:]

    layers = embedder.img_processor.vision_model.encoder.layers
    idx = embedder.layer_idx if embedder.layer_idx >= 0 else len(layers) + embedder.layer_idx
    del layers[idx + 1:]
    del embedder.img_processor.vision_model.post_layernorm
    embedder.get_img_features = get_img_features.__get__(embedder)


def load(adapter=MED_ADAPTER, precision='fp16', crops=16, train=False, revision=None):
    """precision: fp16 (T4), bf16 (A100/L4), 4bit (NF4, low memory).

    Eager attention is required: Expert-CFG adds an additive bias to the 4D attention
    mask (Eq. 9), which flash-attention would silently ignore.
    """
    if not torch.cuda.is_available():
        raise RuntimeError('CUDA GPU required (Colab: Runtime > Change runtime type > GPU).')
    from transformers import AutoModelForCausalLM, AutoProcessor, BitsAndBytesConfig
    from peft import PeftModel, prepare_model_for_kbit_training

    dtype = torch.bfloat16 if precision == 'bf16' else torch.float16
    quant = None
    if precision == '4bit':
        quant = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type='nf4',
                                   bnb_4bit_compute_dtype=torch.float16,
                                   bnb_4bit_use_double_quant=True)
    model = AutoModelForCausalLM.from_pretrained(
        BASE_MODEL, revision=revision, trust_remote_code=True, torch_dtype=dtype,
        quantization_config=quant, device_map={'': 'cuda:0'},
        _attn_implementation='eager')
    processor = AutoProcessor.from_pretrained(BASE_MODEL, revision=revision,
                                              trust_remote_code=True, num_crops=crops)
    patch_clip_for_lora(model)
    if train:
        if quant:
            model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=True)
        else:
            model.requires_grad_(False)
            model.enable_input_require_grads()
        model.config.use_cache = False
    if adapter:
        model = PeftModel.from_pretrained(model, adapter, is_trainable=train)
    if train:
        for name, p in model.named_parameters():
            if 'vision_embed_tokens' in name:  # paper: vision encoder frozen
                p.requires_grad_(False)
    else:
        model.eval()
    meta = {'base_model': BASE_MODEL, 'adapter': adapter, 'precision': precision,
            'num_crops': crops, 'gpu': torch.cuda.get_device_name(),
            'versions': {p: importlib.metadata.version(p) for p in
                         ('torch', 'transformers', 'peft', 'accelerate')}}
    return model, processor, meta


def model_dtype(model):
    for p in model.parameters():
        if p.dtype in (torch.float16, torch.bfloat16, torch.float32):
            return p.dtype
    return torch.float16


In [ ]:
%%writefile /content/phi3_baseline/phi3cfg/generation.py
#@title phi3cfg/generation.py
"""Greedy decoding for Phi-3-Vision with predictive entropy (Eq. 4) and Expert-CFG (Eq. 7-10).

Two-branch batch for Expert-CFG (batch index 0 = normal context c, 1 = unconditional c-bar):
  Eq. 7  c-bar_i = (alpha-1) m_i f(x_i) + f(x_i)      -> highlighted token embeddings x alpha in branch 1
  Eq. 9  h_i = log(beta) m_i + e_i                     -> attention-score bias on highlighted keys in branch 0
         P~(x | c-bar, -delta m)                       -> attention-score bias -delta on highlighted keys in branch 1
  Eq. 8  log P^ = gamma log P(x|c) - (gamma-1) log P(x|c-bar)   -> greedy argmax

The bias is added to the additive 4D attention mask of every Phi-3 decoder self-attention
layer (eager attention), i.e. before softmax, exactly as Eq. 9/10. The CLIP encoder is untouched.
"""
import math
import re
from dataclasses import dataclass

import torch

from .model import model_dtype

IMAGE_TAG = re.compile(r'<\|image_\d+\|>')


@dataclass
class CFGConfig:
    gamma: float = 1.5   # main text; appendix Table 7 best = 1.3
    alpha: float = 0.01
    beta: float = 3.0
    delta: float = None  # paper: "delta satisfying log(beta) + 2"

    def __post_init__(self):
        if self.delta is None:
            self.delta = math.log(self.beta) + 2.0
        if self.gamma < 1 or self.beta <= 0 or not 0 <= self.alpha <= 1:
            raise ValueError('need gamma >= 1, beta > 0, 0 <= alpha <= 1')


def build_prompt(tokenizer, question, reference=None):
    # Same layout as the author demo: f'{caption} <|image_1|>\n{question}'
    content = f'{reference} <|image_1|>\n{question}' if reference else f'<|image_1|>\n{question}'
    return tokenizer.apply_chat_template([{'role': 'user', 'content': content}],
                                         tokenize=False, add_generation_prompt=True)


def _char_ranges(text, parts):
    low = text.lower()
    ranges = []
    for part in parts or []:
        part = part.strip()
        if not part:
            continue
        start = low.find(part.lower())
        while start != -1:
            ranges.append((start, start + len(part)))
            start = low.find(part.lower(), start + len(part))
    return ranges


def highlight_mask(tokenizer, prompt, input_ids, highlights):
    """Token mask m aligned to the processor's input_ids (image placeholder ids are negative).

    Each text chunk between image tags is re-tokenized with character offsets; the processor's
    text segments must equal a suffix of that tokenization (the processor strips BOS), which is
    asserted so a tokenizer/processor mismatch fails loudly instead of misplacing highlights.
    """
    ids = input_ids[0].tolist()
    segments, i = [], 0
    while i < len(ids):
        if ids[i] < 0:
            i += 1
            continue
        j = i
        while j < len(ids) and ids[j] >= 0:
            j += 1
        segments.append((i, j))
        i = j
    chunks = IMAGE_TAG.split(prompt)
    if len(segments) != len(chunks):
        raise ValueError(f'{len(chunks)} text chunks but {len(segments)} token segments')
    mask = torch.zeros(len(ids))
    for (s, e), chunk in zip(segments, chunks):
        enc = tokenizer(chunk, return_offsets_mapping=True)
        tok, offs = enc['input_ids'], enc['offset_mapping']
        n = e - s
        if n > len(tok) or tok[len(tok) - n:] != ids[s:e]:
            raise ValueError('Highlight alignment failed: processor tokenization differs')
        ranges = _char_ranges(chunk, highlights)
        for k, (a, b) in enumerate(offs[len(tok) - n:]):
            if a < b and any(a < r1 and b > r0 for r0, r1 in ranges):
                mask[s + k] = 1.0
    return mask


class _Hooks:
    def __init__(self, model):
        self.embed_scale = None  # (B, P)
        self.attn_bias = None    # (B, P)
        self.handles = []
        embedders = [m for n, m in model.named_modules() if n.endswith('vision_embed_tokens')]
        attns = [m for n, m in model.named_modules()
                 if re.search(r'layers\.\d+\.self_attn$', n) and 'img_processor' not in n]
        if len(embedders) != 1 or not attns:
            raise RuntimeError('Unexpected Phi-3-V module layout')
        self.handles.append(embedders[0].register_forward_hook(self._embed))
        for m in attns:
            self.handles.append(m.register_forward_pre_hook(self._attn, with_kwargs=True))

    def _embed(self, module, args, output):
        if self.embed_scale is None:
            return output
        P = self.embed_scale.shape[1]
        if output.shape[1] < P:
            return output
        output = output.clone()
        output[:, :P] = output[:, :P] * self.embed_scale[:, :, None].to(output.dtype)
        return output

    def _attn(self, module, args, kwargs):
        if self.attn_bias is None:
            return None
        positional = 'attention_mask' not in kwargs
        mask = args[1] if positional and len(args) > 1 else kwargs.get('attention_mask')
        if mask is None or mask.dim() != 4:
            raise RuntimeError('Expected a 4D additive mask: load the model with eager attention')
        kv = mask.shape[-1]
        bias = self.attn_bias
        if kv > bias.shape[1]:
            bias = torch.nn.functional.pad(bias, (0, kv - bias.shape[1]))
        mask = mask + bias[:, None, None, :kv].to(mask.dtype)
        if positional:
            args = (args[0], mask, *args[2:])
        else:
            kwargs['attention_mask'] = mask
        return args, kwargs

    def clear(self):
        self.embed_scale = self.attn_bias = None


_HOOKS = {}


def _hooks(model):
    if id(model) not in _HOOKS:
        _HOOKS[id(model)] = _Hooks(model)
    return _HOOKS[id(model)]


def stop_ids(model, tokenizer):
    ids = {tokenizer.eos_token_id, tokenizer.convert_tokens_to_ids('<|end|>')}
    eos = getattr(model.generation_config, 'eos_token_id', None)
    ids.update(eos if isinstance(eos, (list, tuple)) else [eos])
    return {i for i in ids if isinstance(i, int) and i >= 0}


def _entropy(logp):
    p = logp.exp()
    return float(-torch.where(p > 0, p * logp, torch.zeros_like(p)).sum())


@torch.inference_mode()
def generate(model, processor, image, question, reference=None, highlights=None,
             cfg: CFGConfig = None, max_new_tokens=64, use_cache=True):
    """Greedy decode. cfg=None -> plain MedVLM (optionally with a reference in the prompt)."""
    tok = processor.tokenizer
    prompt = build_prompt(tok, question, reference)
    batch = processor(prompt, [image], return_tensors='pt')
    device = model.get_input_embeddings().weight.device
    input_ids = batch['input_ids'].to(device)
    pixel_values = batch['pixel_values'].to(device, model_dtype(model))
    image_sizes = batch['image_sizes'].to(device)
    P = input_ids.shape[1]
    hooks = _hooks(model)
    n_highlight = 0
    if cfg is not None:
        if not highlights:
            raise ValueError('Expert-CFG needs at least one highlight')
        m = highlight_mask(tok, prompt, input_ids, highlights).to(device)
        n_highlight = int(m.sum())
        if n_highlight == 0:
            raise ValueError(f'No highlight found in prompt: {highlights}')
        hooks.embed_scale = torch.stack([torch.ones_like(m), 1 - (1 - cfg.alpha) * m])
        hooks.attn_bias = torch.stack([math.log(cfg.beta) * m, -cfg.delta * m])
        input_ids = input_ids.repeat(2, 1)
        pixel_values = pixel_values.repeat(2, *[1] * (pixel_values.dim() - 1))
        image_sizes = image_sizes.repeat(2, 1)
    B = input_ids.shape[0]
    stops = stop_ids(model, tok)
    special = set(tok.all_special_ids)
    out_ids, ent, ent_cond, probs = [], [], [], []
    past, seq = None, input_ids
    try:
        for _ in range(max_new_tokens):
            attn = torch.ones(B, seq.shape[1], dtype=torch.long, device=device)
            if past is None:
                # clone: Phi-3-V clamps image placeholder ids in-place
                out = model(input_ids=seq.clone(), pixel_values=pixel_values, image_sizes=image_sizes,
                            attention_mask=attn, use_cache=use_cache)
            else:
                out = model(input_ids=seq[:, -1:], attention_mask=attn,
                            past_key_values=past, use_cache=True)
            if use_cache:
                past = out.past_key_values
            logp = out.logits[:, -1].float().log_softmax(-1)
            if cfg is not None:
                final = (cfg.gamma * logp[0] - (cfg.gamma - 1) * logp[1]).log_softmax(-1)
            else:
                final = logp[0]
            nxt = int(final.argmax())
            out_ids.append(nxt)
            if nxt not in special:
                ent.append(_entropy(final))
                ent_cond.append(_entropy(logp[0]))
                probs.append(float(final[nxt].exp()))
            if nxt in stops:
                break
            seq = torch.cat([seq, torch.full((B, 1), nxt, device=device, dtype=seq.dtype)], 1)
    finally:
        hooks.clear()
    mean = lambda v: sum(v) / len(v) if v else None
    return {'prediction': tok.decode(out_ids, skip_special_tokens=True).strip(),
            'entropy': mean(ent),            # Eq. 4 on the decoding distribution
            'entropy_normal_branch': mean(ent_cond),
            'token_probs': probs,
            'tokens': [tok.decode([i]) for i in out_ids if i not in special],
            'truncated': bool(out_ids) and out_ids[-1] not in stops,
            'prompt_tokens': P, 'highlight_tokens': n_highlight}


In [ ]:
%%writefile /content/phi3_baseline/phi3cfg/infer.py
#@title phi3cfg/infer.py
"""Run Phi3V-Med on a MedVQA split in one of the Table 1 modes (resumable JSONL output).

  baseline    plain Phi3V-Med; records Eq. 4 entropy (needed to pick the top-x% samples)
  rag         top-1 retrieved caption in the prompt
  expert_rag  expert reference (retrieved caption + expert text) in the prompt, no guidance
  expert_cfg  expert reference + highlights + classifier-free guidance (Eq. 7-10)

With --select-from BASELINE.jsonl --top-percent 5 only the top-5% highest-entropy questions
are run (the rest keep their baseline answers, merged by phi3cfg.evaluate).
"""
import argparse
import json
from pathlib import Path

from PIL import Image

from .data import load_data, read_jsonl
from .model import MED_ADAPTER


def select_top(baseline_path, percent=None, threshold=None):
    rows = [r for r in read_jsonl(baseline_path) if r.get('entropy') is not None]
    rows.sort(key=lambda r: r['entropy'], reverse=True)
    if threshold is not None:
        return {r['id'] for r in rows if r['entropy'] >= threshold}
    k = max(1, round(len(rows) * percent / 100))
    return {r['id'] for r in rows[:k]}


def main():
    p = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)
    p.add_argument('--dataset-dir', required=True, help='DATA/<vqa_rad|slake|pathvqa>')
    p.add_argument('--split', default='test')
    p.add_argument('--mode', choices=('baseline', 'rag', 'expert_rag', 'expert_cfg'), default='baseline')
    p.add_argument('--annotations', help='output of phi3cfg.retrieval search (non-baseline modes)')
    p.add_argument('--select-from', help='baseline predictions; restrict to high-entropy questions')
    p.add_argument('--top-percent', type=float)
    p.add_argument('--entropy-threshold', type=float)
    p.add_argument('--output', required=True)
    p.add_argument('--adapter', default=MED_ADAPTER,
                   help='HF id or local downstream LoRA dir (phi3cfg.train output); "none" = raw Phi-3V')
    p.add_argument('--precision', choices=('fp16', 'bf16', '4bit'), default='fp16')
    p.add_argument('--crops', type=int, default=16)
    p.add_argument('--max-new-tokens', type=int, default=64)
    p.add_argument('--no-cache', action='store_true', help='recompute full context each step')
    p.add_argument('--gamma', type=float, default=1.5)
    p.add_argument('--alpha', type=float, default=0.01)
    p.add_argument('--beta', type=float, default=3.0)
    p.add_argument('--delta', type=float, help='default log(beta)+2')
    p.add_argument('--limit', type=int)
    args = p.parse_args()
    if args.mode != 'baseline' and not args.annotations:
        p.error('--annotations is required for rag/expert modes')
    if args.select_from and args.top_percent is None and args.entropy_threshold is None:
        p.error('--select-from needs --top-percent or --entropy-threshold')

    ds_dir = Path(args.dataset_dir)
    rows = load_data(ds_dir / f'{args.split}.jsonl', args.split)
    if args.select_from:
        keep = select_top(args.select_from, args.top_percent, args.entropy_threshold)
        rows = [r for r in rows if r['id'] in keep]
        print(f'Selected {len(rows)} high-entropy questions', flush=True)
    if args.limit:
        rows = rows[:args.limit]
    ann = {a['id']: a for a in read_jsonl(args.annotations)} if args.annotations else {}

    from .generation import CFGConfig, generate
    from .model import load
    adapter = None if args.adapter.lower() == 'none' else args.adapter
    model, processor, meta = load(adapter, args.precision, args.crops)
    cfg = CFGConfig(args.gamma, args.alpha, args.beta, args.delta)

    out = Path(args.output)
    out.parent.mkdir(parents=True, exist_ok=True)
    meta_path = out.with_suffix('.meta.json')
    config = {**vars(args), 'runtime': meta, 'cfg': vars(cfg)}
    if meta_path.exists() and json.loads(meta_path.read_text())['mode'] != args.mode:
        raise ValueError('Output belongs to another mode; choose a new --output')
    meta_path.write_text(json.dumps(config, indent=2))
    done = {r['id'] for r in read_jsonl(out)} if out.exists() else set()

    with out.open('a', encoding='utf-8') as f:
        for n, r in enumerate(rows, 1):
            if r['id'] in done:
                continue
            a = ann.get(r['id'], {})
            if args.mode != 'baseline' and not a:
                raise ValueError(f'No annotation for {r["id"]}')
            with Image.open(ds_dir / 'images' / r['image']) as im:
                image = im.convert('RGB')
            kw = dict(max_new_tokens=args.max_new_tokens, use_cache=not args.no_cache)
            cfg_applied = False
            if args.mode == 'baseline':
                res = generate(model, processor, image, r['question'], **kw)
            elif args.mode == 'rag':
                res = generate(model, processor, image, r['question'], a['rag_reference'], **kw)
            elif args.mode == 'expert_rag':
                res = generate(model, processor, image, r['question'], a['expert_reference'], **kw)
            else:
                hl = a.get('highlights') or []
                cfg_applied = bool(hl)
                try:
                    res = generate(model, processor, image, r['question'], a['expert_reference'],
                                   hl, cfg if hl else None, **kw)
                except ValueError as e:  # highlight not tokenizable in prompt -> no guidance
                    print(f'  {r["id"]}: {e}; falling back to expert_rag', flush=True)
                    cfg_applied = False
                    res = generate(model, processor, image, r['question'], a['expert_reference'], **kw)
            rec = {**r, **res, 'mode': args.mode, 'cfg_applied': cfg_applied}
            f.write(json.dumps(rec, ensure_ascii=False) + '\n')
            f.flush()
            print(f'[{n}/{len(rows)}] H={res["entropy"] if res["entropy"] is None else round(res["entropy"], 3)} '
                  f'Q: {r["question"][:60]} | GT: {r["answer"]} | P: {res["prediction"][:80]}', flush=True)
    print('Predictions:', out)


if __name__ == '__main__':
    main()


In [ ]:
%%writefile /content/phi3_baseline/phi3cfg/evaluate.py
#@title phi3cfg/evaluate.py
"""Score runs and print a Table-1-style comparison for Phi3V-Med (paper values included).

Metrics (Sec. 4.3, LLaVA-Med convention): closed = accuracy (answer word contained in the
prediction), open = recall of ground-truth tokens in the generated answer, overall = mean over
all questions. Open/closed come from the dataset labels (VQA-RAD: run phi3cfg.vqarad_labels).
Partial runs (top-x% modes) are merged over the baseline: questions not re-run keep the baseline answer.

  python -m phi3cfg.evaluate --dataset-dir DATA/vqa_rad --baseline out/base.jsonl \
      --runs rag=out/rag.jsonl rag5=out/rag5.jsonl expert_rag5=out/erag5.jsonl expert_cfg5=out/ecfg5.jsonl \
      --output out/report.json
"""
import argparse
import json
import re
from collections import Counter
from pathlib import Path

from .data import load_data, read_jsonl

# Table 1, Phi3V-Med rows: (open, closed, overall)
PAPER = {
    'vqa_rad': {'baseline': (70.39, 85.29, 79.38), 'rag': (66.48, 90.07, 80.71),
                'rag5': (68.72, 85.29, 78.71), 'expert_rag5': (71.50, 86.03, 80.27),
                'expert_cfg5': (75.42, 86.03, 81.82)},
    'slake': {'baseline': (82.02, 87.02, 83.98), 'rag': (83.41, 87.74, 85.11),
              'rag5': (82.33, 87.50, 84.35), 'expert_rag5': (83.25, 88.46, 85.29),
              'expert_cfg5': (84.96, 88.46, 86.33)},
    'pathvqa': {'baseline': (36.88, 91.80, 64.43), 'rag': (49.05, 95.28, 72.24),
                'rag5': (38.81, 91.80, 65.39), 'expert_rag5': (42.37, 91.80, 67.16),
                'expert_cfg5': (45.88, 91.80, 68.91)},
}


def tokens(text):
    return re.findall(r'[a-z0-9]+', str(text).lower())


def score(answer, prediction, answer_type):
    """Same rule as the team's Phi-3.5V baseline (MinseongKim-dev/Expert-CFG, 01 notebook):
    closed = the whole lower-cased answer is one of the prediction's tokens (so a multi-word
    closed answer such as "right side" never scores); open = recall of answer tokens."""
    pred = tokens(prediction)
    if answer_type == 'closed':
        return float(str(answer).strip().lower() in pred)
    counts = Counter(tokens(answer))
    return sum((counts & Counter(pred)).values()) / sum(counts.values()) if counts else 0.0


def summarize(rows, preds):
    s = {'open': [], 'closed': []}
    for r in rows:
        s[r['answer_type']].append(score(r['answer'], preds[r['id']]['prediction'], r['answer_type']))
    pct = lambda v: 100 * sum(v) / len(v) if v else float('nan')
    return {'open': pct(s['open']), 'closed': pct(s['closed']),
            'overall': pct(s['open'] + s['closed']),
            'n_open': len(s['open']), 'n_closed': len(s['closed'])}


def uncertainty_auc(rows, preds):
    from sklearn.metrics import roc_auc_score
    y, h = [], []
    for r in rows:
        p = preds[r['id']]
        if p.get('entropy') is None:
            continue
        y.append(int(score(r['answer'], p['prediction'], r['answer_type']) < 1.0))  # 1 = wrong
        h.append(p['entropy'])
    return float(roc_auc_score(y, h)) if len(set(y)) == 2 else None


def main():
    p = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)
    p.add_argument('--dataset-dir', required=True)
    p.add_argument('--split', default='test')
    p.add_argument('--baseline', required=True)
    p.add_argument('--runs', nargs='*', default=[], help='name=path (names: rag, rag5, expert_rag5, expert_cfg5, ...)')
    p.add_argument('--output')
    p.add_argument('--partial', action='store_true', help='score only questions in the baseline file')
    args = p.parse_args()
    rows = load_data(Path(args.dataset_dir) / f'{args.split}.jsonl', args.split)
    dataset = rows[0]['dataset']
    base = {r['id']: r for r in read_jsonl(args.baseline)}
    missing = [r['id'] for r in rows if r['id'] not in base]
    if missing and args.partial:
        print(f'WARNING: scoring only {len(rows) - len(missing)}/{len(rows)} questions (smoke test)')
        rows = [r for r in rows if r['id'] in base]
    elif missing:
        raise SystemExit(f'Baseline is missing {len(missing)} questions (e.g. {missing[:3]}); finish it first')

    results = {'baseline': {**summarize(rows, base), 'replaced': 0}}
    for spec in args.runs:
        name, path = spec.split('=', 1)
        run = {r['id']: r for r in read_jsonl(path)}
        merged = {i: run.get(i, b) for i, b in base.items()}
        results[name] = {**summarize(rows, merged), 'replaced': sum(i in run for i in base)}
        if any('cfg_applied' in r for r in run.values()):
            results[name]['cfg_applied'] = sum(bool(r.get('cfg_applied')) for r in run.values())
    auc = uncertainty_auc(rows, base)

    paper = PAPER.get(dataset, {}) if args.split == 'test' else {}  # paper numbers are test-set numbers
    print(f'\n{dataset} ({len(rows)} questions)   uncertainty AUC (baseline entropy) = {auc}')
    print(f'{"run":<14}{"open":>8}{"closed":>8}{"overall":>9}   | paper open/closed/overall | re-run')
    for name, m in results.items():
        ref = paper.get(name)
        ref_s = '/'.join(f'{v:.2f}' for v in ref) if ref else '-'
        print(f'{name:<14}{m["open"]:8.2f}{m["closed"]:8.2f}{m["overall"]:9.2f}   | {ref_s:<25} | {m["replaced"]}')
    if args.output:
        Path(args.output).parent.mkdir(parents=True, exist_ok=True)
        Path(args.output).write_text(json.dumps({'dataset': dataset, 'uncertainty_auc': auc,
                                                 'results': results, 'paper': paper}, indent=2))


if __name__ == '__main__':
    main()


In [ ]:
%%writefile /content/phi3_baseline/phi3cfg/vqarad_labels.py
#@title phi3cfg/vqarad_labels.py
"""Replace the rule-based VQA-RAD answer_type with the original dataset labels.

The HF mirror (flaviagiammarino/vqa-rad) has no answer_type column, so download_data falls back
to "yes/no answer -> closed" (test: closed 251 / open 200). The original VQA-RAD labels also mark
choice questions such as "left or right?" as CLOSED (test: closed 272 / open 179), which is the
convention the paper's Open/Closed numbers use.

Label source: the original VQA-RAD questions (all 3,515, with answer_type) as published in the
team repo MinseongKim-dev/Expert-CFG, pinned to one commit and checked by sha256. Rows are
matched on normalized (question, answer); the test split must match completely and give 272/179.

  python -m phi3cfg.vqarad_labels --dataset-dir DATA/vqa_rad [--label-dir DIR]

Idempotent: provenance.json records answer_type_source and the run is skipped afterwards.
Only answer_type changes (old value kept as answer_type_rule), so training is unaffected.
"""
import argparse
import hashlib
import json
import re
import urllib.request
from collections import defaultdict
from pathlib import Path

from .data import digest, load_data

COMMIT = '6912f3b440ad9643d6c8d034332f65c498ffba9a'
URL = f'https://raw.githubusercontent.com/MinseongKim-dev/Expert-CFG/{COMMIT}/data_common/vqarad_img/{{}}'
FILES = {  # original train (3,064 = trainset + valset) and test (451)
    'trainset.json': ('train', '97b7758929bc3c17b101c3be8eadaea3a82cba2844d6c7274763ba3149c65108'),
    'valset.json': ('train', 'b053fb3996e09c0f34e2321805a406f5359f317f2eae0993c30fb55d3c55a353'),
    'testset.json': ('test', '3d32a047592cb680e18054030b38b1a44d6b17105fc62d3088c8655d56a00ced'),
}
EXPECTED_TEST = {'closed': 272, 'open': 179}
SOURCE = 'original_vqarad'


def norm(text):
    return ' '.join(re.sub(r'[^a-z0-9]+', ' ', str(text).lower()).split())


def fetch_labels(label_dir):
    label_dir = Path(label_dir)
    label_dir.mkdir(parents=True, exist_ok=True)
    for name, (_, sha) in FILES.items():
        path = label_dir / name
        if not path.exists():
            print(f'Downloading {name} (Expert-CFG@{COMMIT[:7]})', flush=True)
            with urllib.request.urlopen(URL.format(name), timeout=60) as r:
                data = r.read()
            if hashlib.sha256(data).hexdigest() != sha:
                raise ValueError(f'{name}: sha256 mismatch for downloaded label file')
            path.write_bytes(data)
        elif digest(path) != sha:
            raise ValueError(f'{path}: sha256 mismatch; delete it to re-download')
    return label_dir


def build_index(label_dir):
    index = defaultdict(set)  # (part|'any', question, answer) -> {answer_type}
    for name, (part, _) in FILES.items():
        for r in json.loads((Path(label_dir) / name).read_text(encoding='utf-8')):
            kind = str(r['answer_type']).strip().lower()  # source has values like 'CLOSED '
            if kind not in ('open', 'closed'):
                raise ValueError(f'{name}: unexpected answer_type {r["answer_type"]!r}')
            key = (norm(r['question']), norm(r['answer']))
            index[(part, *key)].add(kind)
            index[('any', *key)].add(kind)
    return index


def relabel(dataset_dir, label_dir=None):
    d = Path(dataset_dir)
    prov_path = d / 'provenance.json'
    prov = json.loads(prov_path.read_text(encoding='utf-8'))
    if prov.get('dataset') != 'vqa_rad':
        raise ValueError(f'{d} is not the vqa_rad dataset')
    if prov.get('answer_type_source') == SOURCE:
        print(f'{d}: answer_type already uses original VQA-RAD labels', flush=True)
        return prov['answer_type_relabel']
    index = build_index(fetch_labels(label_dir or d.parent / 'vqarad_original_labels'))

    updated, report = {}, {}
    for split in prov['splits']:
        rows = load_data(d / f'{split}.jsonl', split)
        part = 'test' if split == 'test' else 'train'
        unmatched, ambiguous, changed = [], [], 0
        for r in rows:
            key = (norm(r['question']), norm(r['answer']))
            kinds = index.get((part, *key)) or index.get(('any', *key))
            if not kinds:
                unmatched.append(r['id'])
                continue
            if len(kinds) > 1:
                ambiguous.append(r['id'])
                continue
            kind = next(iter(kinds))
            if kind != r['answer_type']:
                r['answer_type_rule'] = r['answer_type']
                r['answer_type'] = kind
                changed += 1
        closed = sum(r['answer_type'] == 'closed' for r in rows)
        updated[split] = rows
        report[split] = {'n': len(rows), 'closed': closed, 'open': len(rows) - closed, 'changed': changed,
                         'unmatched': len(unmatched), 'ambiguous': len(ambiguous),
                         'unmatched_ids': unmatched[:20], 'ambiguous_ids': ambiguous[:20]}
        print(f'vqa_rad/{split}: closed {closed} / open {len(rows) - closed} '
              f'(changed {changed}, unmatched {len(unmatched)}, ambiguous {len(ambiguous)})', flush=True)

    t = report['test']
    if t['unmatched'] or t['ambiguous'] or {k: t[k] for k in EXPECTED_TEST} != EXPECTED_TEST:
        raise ValueError(f'Test labels incomplete or counts differ from {EXPECTED_TEST}: {t}. Nothing was written.')

    temps = []
    for split, rows in updated.items():
        tmp = d / f'{split}.jsonl.tmp'
        tmp.write_text(''.join(json.dumps(r, ensure_ascii=False) + '\n' for r in rows), encoding='utf-8')
        temps.append((tmp, d / f'{split}.jsonl'))
    for tmp, path in temps:
        tmp.replace(path)
        prov['files'][path.name] = digest(path)
        s = report[path.name[:-len('.jsonl')]]
        prov['splits'][path.name[:-len('.jsonl')]].update(open=s['open'], closed=s['closed'])
    prov['answer_type_source'] = SOURCE
    prov['answer_type_label_files'] = {'repo': 'MinseongKim-dev/Expert-CFG', 'commit': COMMIT,
                                       'sha256': {n: sha for n, (_, sha) in FILES.items()}}
    prov['answer_type_relabel'] = report
    tmp = prov_path.with_suffix('.tmp')
    tmp.write_text(json.dumps(prov, indent=2, ensure_ascii=False) + '\n', encoding='utf-8')
    tmp.replace(prov_path)
    return report


def main():
    p = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)
    p.add_argument('--dataset-dir', required=True, help='prepared DATA/vqa_rad (download_data output)')
    p.add_argument('--label-dir', help='where the original label JSONs are cached (default: DATA/vqarad_original_labels)')
    args = p.parse_args()
    relabel(args.dataset_dir, args.label_dir)


if __name__ == '__main__':
    main()


In [ ]:
%%writefile /content/phi3_baseline/phi3cfg/train.py
#@title phi3cfg/train.py
"""Downstream fine-tuning of Phi3V-Med on one MedVQA train split (Sec. 4.3).

Paper: LoRA rank 64 only, AdamW + cosine, lr 1e-4, batch 64, 10 epochs, vision encoder frozen.
Starts from the released Phi-3V-Med LoRA (ecoxial2007/Phi-3V-Med, r=64, alpha=128).
Phi-3-V supports micro-batch 1, so batch 64 = 1 x 64 gradient accumulation.

  python -m phi3cfg.train --dataset-dir DATA/vqa_rad --output runs/phi3v_vqa_rad
"""
import argparse
import json
from pathlib import Path

import torch
from PIL import Image

from .data import load_data
from .generation import build_prompt
from .model import MED_ADAPTER, load, model_dtype


def main():
    p = argparse.ArgumentParser()
    p.add_argument('--dataset-dir', required=True)
    p.add_argument('--output', required=True)
    p.add_argument('--adapter', default=MED_ADAPTER)
    p.add_argument('--precision', choices=('fp16', 'bf16', '4bit'), default='fp16')
    p.add_argument('--crops', type=int, default=16, help='lower (e.g. 4) if T4 runs out of memory')
    p.add_argument('--epochs', type=int, default=10)
    p.add_argument('--lr', type=float, default=1e-4)
    p.add_argument('--batch', type=int, default=64)
    p.add_argument('--seed', type=int, default=42)
    p.add_argument('--resume', action='store_true')
    p.add_argument('--save-epochs', help='e.g. "2,4,6,8,10": checkpoint only at these epochs and keep them all '
                                         '(for validation-based selection). Default: every epoch, keep last 2')
    p.add_argument('--no-grad-ckpt', action='store_true', help='disable gradient checkpointing (faster, more memory)')
    args = p.parse_args()
    save_epochs = {int(e) for e in args.save_epochs.split(',')} if args.save_epochs else None
    if save_epochs and not all(1 <= e <= args.epochs for e in save_epochs):
        p.error(f'--save-epochs must lie in 1..{args.epochs}')

    from transformers import Trainer, TrainerCallback, TrainingArguments, set_seed
    set_seed(args.seed)
    ds_dir = Path(args.dataset_dir)
    train = load_data(ds_dir / 'train.jsonl', 'train')
    model, processor, meta = load(args.adapter, args.precision, args.crops, train=True)
    lora = next(iter(model.peft_config.values()))
    assert lora.r == 64, f'expected LoRA rank 64, got {lora.r}'
    for prm in model.parameters():  # AMP needs fp32 master weights for trainable params
        if prm.requires_grad:
            prm.data = prm.data.float()
    model.print_trainable_parameters()
    tok = processor.tokenizer
    dtype = model_dtype(model)

    def collate(rows):
        assert len(rows) == 1, 'Phi-3-V supports micro-batch 1'
        r = rows[0]
        with Image.open(ds_dir / 'images' / r['image']) as im:
            batch = processor(build_prompt(tok, r['question']), [im.convert('RGB')], return_tensors='pt')
        ans = tok(f"{r['answer']}<|end|>\n<|endoftext|>", add_special_tokens=False,
                  return_tensors='pt')['input_ids']
        batch['labels'] = torch.cat([torch.full_like(batch['input_ids'], -100), ans], 1)
        batch['input_ids'] = torch.cat([batch['input_ids'], ans], 1)
        batch['attention_mask'] = torch.ones_like(batch['input_ids'])
        batch['pixel_values'] = batch['pixel_values'].to(dtype)
        return dict(batch)

    out = Path(args.output)
    out.mkdir(parents=True, exist_ok=True)
    (out / 'run_config.json').write_text(json.dumps({**vars(args), 'runtime': meta}, indent=2))
    targs = TrainingArguments(
        output_dir=str(out), num_train_epochs=args.epochs, per_device_train_batch_size=1,
        gradient_accumulation_steps=args.batch, learning_rate=args.lr, lr_scheduler_type='cosine',
        optim='adamw_torch', weight_decay=0.0, warmup_ratio=0.03,
        fp16=args.precision != 'bf16', bf16=args.precision == 'bf16',
        gradient_checkpointing=not args.no_grad_ckpt,
        gradient_checkpointing_kwargs={'use_reentrant': False} if not args.no_grad_ckpt else None,
        remove_unused_columns=False, dataloader_num_workers=2, logging_steps=5,
        save_strategy='epoch', save_total_limit=None if save_epochs else 2,
        report_to='none', seed=args.seed)

    class SaveAtEpochs(TrainerCallback):
        """Runs after the default flow callback: keep its per-epoch save only at the chosen epochs."""
        def on_epoch_end(self, args, state, control, **kwargs):
            control.should_save = int(round(state.epoch)) in save_epochs
            return control

    trainer = Trainer(model=model, args=targs, train_dataset=train, data_collator=collate,
                      callbacks=[SaveAtEpochs()] if save_epochs else None)
    if args.resume:
        # torch>=2.6 defaults torch.load(weights_only=True), which rejects the numpy RNG state that
        # transformers 4.44 saves in rng_state.pth. These checkpoints are our own, so load them fully.
        _load = torch.load
        torch.load = lambda *a, **k: _load(*a, **{**k, 'weights_only': False})
    trainer.train(resume_from_checkpoint=True if args.resume else None)
    trainer.save_model(str(out))  # adapter_config.json + adapter_model.safetensors
    print('Downstream LoRA saved to', out, '-> use as --adapter for phi3cfg.infer')


if __name__ == '__main__':
    main()


In [11]:
os.chdir(CODE)
LIM = f'--limit {LIMIT}' if LIMIT else ''

## 4. 저장된 진행 상황 확인
데이터, 체크포인트, 결과 파일을 점검한다. 마지막 체크포인트가 불완전하면 여기서 `broken-`으로 이름을 바꾼다.

In [12]:
import glob, json, os, re, torch

REQUIRED = ('trainer_state.json', 'adapter_model.safetensors', 'optimizer.pt', 'scheduler.pt')

def ckpt_ok(path):
    missing = [f for f in REQUIRED if not os.path.exists(f'{path}/{f}')]
    if not missing:
        try:
            json.load(open(f'{path}/trainer_state.json'))
        except Exception as e:
            missing = [f'trainer_state.json 읽기 실패: {e}']
    return missing

def checkpoints(out):
    cs = [c for c in glob.glob(f'{out}/checkpoint-*') if re.fullmatch(r'checkpoint-\d+', os.path.basename(c))]
    return sorted(cs, key=lambda c: int(c.rsplit('-', 1)[1]))

assert os.path.exists(f'{DATA}/data_paths.json'), f'{DATA}/data_paths.json 없음 - 원래 노트북의 데이터 다운로드부터 실행'
print('GPU:', torch.cuda.get_device_name(0), '| bf16 지원:', torch.cuda.is_bf16_supported())

for ds in DATASETS:
    out = f'{WORK}/lora_{ds}'
    print(f'\n== {ds} ==')
    if os.path.exists(f'{out}/adapter_model.safetensors'):
        print('  학습 완료 (adapter_model.safetensors 있음)')
        continue
    cs = checkpoints(out)
    while cs and (missing := ckpt_ok(cs[-1])):
        bad = cs.pop()
        os.rename(bad, f'{out}/broken-{os.path.basename(bad)}')
        print(f'  {os.path.basename(bad)} 불완전 {missing} -> broken-{os.path.basename(bad)} 로 이동')
    if not cs:
        print('  체크포인트 없음 -> 처음부터 학습')
        continue
    s = json.load(open(f'{cs[-1]}/trainer_state.json'))
    print(f"  재개 지점: {os.path.basename(cs[-1])}  step {s['global_step']}/{s['max_steps']}  "
          f"epoch {s['epoch']:.2f}/{s['num_train_epochs']}")
    for h in [h for h in s['log_history'] if 'loss' in h][-5:]:
        print(f"    step {h['step']:>5}  epoch {h['epoch']:.2f}  loss {h['loss']:.4f}  lr {h.get('learning_rate', 0):.2e}")
    if os.path.exists(f'{out}/run_config.json'):
        rc = json.load(open(f'{out}/run_config.json'))
        print('  run_config:', {k: rc[k] for k in ('precision', 'crops', 'epochs', 'lr', 'batch', 'seed') if k in rc})

GPU: NVIDIA A100-SXM4-80GB | bf16 지원: True

== vqa_rad ==
  재개 지점: checkpoint-84  step 84/280  epoch 3.00/10
    step    60  epoch 2.14  loss 0.1126  lr 9.15e-05
    step    65  epoch 2.32  loss 0.0701  lr 8.98e-05
    step    70  epoch 2.50  loss 0.0857  lr 8.80e-05
    step    75  epoch 2.68  loss 0.0726  lr 8.61e-05
    step    80  epoch 2.86  loss 0.0867  lr 8.40e-05
  run_config: {'precision': 'fp16', 'crops': 16, 'epochs': 10, 'lr': 0.0001, 'batch': 64, 'seed': 42}

== slake ==
  체크포인트 없음 -> 처음부터 학습


## 5. 학습 재개
체크포인트가 있으면 `--resume`으로 이어서, 없으면 처음부터 학습한다. 다시 끊기면 이 노트북을 처음부터 다시 실행하면 된다.

In [13]:
ADAPTERS = {}
for ds in DATASETS:
    out = f'{WORK}/lora_{ds}'
    ADAPTERS[ds] = out
    if os.path.exists(f'{out}/adapter_model.safetensors'):
        print(f'{ds}: 학습 완료, 건너뜀')
        continue
    resume = bool(checkpoints(out))
    if resume:
        rc = json.load(open(f'{out}/run_config.json'))
        prec, crops, epochs = rc['precision'], rc['crops'], rc['epochs']
        extra = f"--lr {rc['lr']} --batch {rc['batch']} --seed {rc['seed']} --resume"
        if rc.get('adapter'):
            extra += f" --adapter {rc['adapter']}"
        if prec == 'bf16' and not torch.cuda.is_bf16_supported():
            raise RuntimeError(f'{ds}: 원래 bf16으로 학습했는데 현재 GPU는 bf16 미지원. A100/L4 런타임으로 바꿔서 실행')
    else:
        prec = NEW_PREC or ('bf16' if torch.cuda.is_bf16_supported() else 'fp16')
        crops, epochs, extra = NEW_TRAIN_CROPS, NEW_EPOCHS, ''
    print(f'{ds}: {"재개" if resume else "새로 학습"}  precision={prec} crops={crops} epochs={epochs}')
    !python -m phi3cfg.train --dataset-dir {DATA}/{ds} --output {out} --precision {prec} --crops {crops} --epochs {epochs} {extra}
    if not os.path.exists(f'{out}/adapter_model.safetensors'):
        raise RuntimeError(f'{ds}: 학습이 끝나지 않음 (위 로그 확인). 다시 실행하면 마지막 체크포인트에서 이어짐')
print(ADAPTERS)

vqa_rad: 재개  precision=fp16 crops=16 epochs=10
2026-11-15 01:55:10.504965: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-11-15 01:55:10.523849: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1794707710.546500    1051 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1794707710.554246    1051 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1794707710.573512    1051 computation_placer.cc:177] computation placer already regi

RuntimeError: vqa_rad: 학습이 끝나지 않음 (위 로그 확인). 다시 실행하면 마지막 체크포인트에서 이어짐

## 5-1. VQA-RAD 라벨 교체 (원본 answer_type)
학습과 무관하다. 평가 전에 한 번만 실행되면 된다.

In [ ]:
# VQA-RAD: HF mirror에는 answer_type이 없어 "yes/no면 closed" 규칙(test 251/200)이 들어가 있다.
# 원본 VQA-RAD 라벨(test closed 272 / open 179)로 교체한다. 이미 교체했으면 건너뛴다. 학습에는 영향 없음.
if 'vqa_rad' in DATASETS:
    !python -m phi3cfg.vqarad_labels --dataset-dir {DATA}/vqa_rad


## 6. Test 추론 (greedy, entropy 기록)

In [ ]:
TAG = 'finetuned'
LIM = f'--limit {LIMIT}' if LIMIT else ''
for ds in DATASETS:
    prec = json.load(open(f'{ADAPTERS[ds]}/run_config.json'))['precision']
    if prec == 'bf16' and not torch.cuda.is_bf16_supported():
        prec = 'fp16'
    os.makedirs(f'{WORK}/{ds}', exist_ok=True)
    !python -m phi3cfg.infer --dataset-dir {DATA}/{ds} --adapter {ADAPTERS[ds]} --precision {prec} {LIM} --mode baseline --output {WORK}/{ds}/baseline_{TAG}.jsonl

## 7. 평가 (논문 값과 비교)

In [ ]:
PARTIAL = '--partial' if LIMIT else ''
for ds in DATASETS:
    !python -m phi3cfg.evaluate --dataset-dir {DATA}/{ds} --baseline {WORK}/{ds}/baseline_{TAG}.jsonl --output {WORK}/{ds}/report_{TAG}.json {PARTIAL}

In [ ]:
import json
print(f"{'dataset':<9}{'':>10}{'open':>8}{'closed':>8}{'overall':>9}   AUC")
for ds in DATASETS:
    r = json.load(open(f'{WORK}/{ds}/report_{TAG}.json'))
    m, p = r['results']['baseline'], r['paper']['baseline']
    print(f"{ds:<9}{'ours':>10}{m['open']:8.2f}{m['closed']:8.2f}{m['overall']:9.2f}   {r['uncertainty_auc']}")
    print(f"{'':<9}{'paper':>10}{p[0]:8.2f}{p[1]:8.2f}{p[2]:9.2f}")
    print(f"{'':<9}{'diff(pp)':>10}{m['open']-p[0]:8.2f}{m['closed']-p[1]:8.2f}{m['overall']-p[2]:9.2f}   (n_open={m['n_open']}, n_closed={m['n_closed']})")

## 8. 오답 예시

In [ ]:
import sys
sys.path.insert(0, CODE)
from phi3cfg.evaluate import score
for ds in DATASETS:
    labels = {r['id']: r['answer_type'] for r in map(json.loads, open(f'{DATA}/{ds}/test.jsonl', encoding='utf-8'))}
    rows = [{**r, 'answer_type': labels[r['id']]} for r in map(json.loads, open(f'{WORK}/{ds}/baseline_{TAG}.jsonl', encoding='utf-8'))]
    wrong = [r for r in rows if score(r['answer'], r['prediction'], r['answer_type']) < 1]
    print(f'== {ds}: {len(wrong)}/{len(rows)} not fully correct')
    for r in wrong[:8]:
        print(f"  [{r['answer_type']}] H={r['entropy']:.3f} Q: {r['question']} | GT: {r['answer']} | P: {r['prediction']}")